# Notebook 12. Surface-Pressure-Screened `850 hPa` Temperature-Gradient Sensitivity

This notebook replaces the earlier terrain-height masking attempt with a pressure-space screen that is better matched to the physical question:

- is the local `850 hPa` level too close to the surface to trust as a clean free-atmosphere frontality diagnostic?

Why this notebook exists:

- the unmasked `850 hPa` temperature-gradient composite in `Notebook 10` still looked strongly terrain-linked even after display-only colorbar tightening
- simply **dropping** the `T850` feature in `Notebook 11` changed too much of the `k = 3` clustering story
- the earlier fixed terrain-height mask attempt did not produce a credible mask source in this workflow
- so the next conservative test is to **keep the feature**, but rebuild it with a surface-pressure screen

What changes here:

- only the `front_box_max_temp_gradient_850_tminus12_to_tplus12` feature is rebuilt
- grid cells with `surface_pressure < 900 hPa` are masked to `NaN` before the front-box maximum is computed
- the other three clustering features stay unchanged

What does **not** change here:

- `925 hPa` signed-divergence feature definitions stay unchanged
- `850 hPa` `z` anomaly logic stays unchanged
- `925 hPa` Sea-of-Japan vorticity stays unchanged
- no `300 hPa` climatology or composite rebuild is done in this notebook

Checkpointing note:

- the event-level pressure-screened `T850` feature table is checkpointed to Drive while it is being built so that a dropped connection does not lose all progress
- the notebook includes a one-event sanity check before the long event loop so a bad mask setup should fail fast


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from itertools import permutations
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import HOKKAIDO_FRONT_BOX, OBJECTIVE_SUBTYPE_DOMAIN
from jpcz_catalog.diagnostics import compute_temperature_gradient_magnitude, load_offset_snapshot
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.subtypes import (
    assign_hierarchical_clusters,
    compute_mean_silhouette_score,
    evaluate_hierarchical_cluster_solutions,
    standardize_feature_table,
)

RUN_EXPORT_DIR_08 = Path("outputs/verification/objective_subtype_runs")
RUN_EXPORT_DIR_08.mkdir(parents=True, exist_ok=True)
SENSITIVITY_EXPORT_DIR = Path("outputs/verification/objective_subtype_t850_surface_pressure_sensitivity")
SENSITIVITY_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_t850_surface_pressure_sensitivity_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CLUSTERED_K3_PATH = RUN_EXPORT_DIR_08 / "clustered_events_k3.csv"
PRIMARY_CLUSTER_COLUMN = "cluster_ward_3"
PRIMARY_CLUSTER_K = 3
SCREENED_CLUSTER_COLUMN_RAW = "cluster_ward_3_screened_t850_sp900hpa_raw"
SCREENED_CLUSTER_COLUMN = "cluster_ward_3_screened_t850_sp900hpa"
DROPPED_FEATURE = "front_box_max_temp_gradient_850_tminus12_to_tplus12"
SCREENED_FEATURE_COLUMN = "front_box_max_temp_gradient_850_tminus12_to_tplus12_screened_sp900hpa"

FULL_FEATURE_COLUMNS = [
    "coastal_to_jpcz_mean_divergence_ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
    DROPPED_FEATURE,
    "sea_of_japan_mean_vorticity_peak_925",
]
SCREENED_FEATURE_COLUMNS = [
    "coastal_to_jpcz_mean_divergence_ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
    SCREENED_FEATURE_COLUMN,
    "sea_of_japan_mean_vorticity_peak_925",
]
CLUSTER_COUNT_OPTIONS = [2, 3, 4]
SAVE_PLOTS = True
ERA5_TIME_CHUNK = 48
FORCE_REBUILD_SCREENED_T850 = False
CHECKPOINT_EVERY_EVENTS = 5
OFFSET_HOURS = (-12, 0, 12)
PRESSURE_SCREEN_THRESHOLD_PA = 90000.0
PRESSURE_SCREEN_THRESHOLD_HPA = PRESSURE_SCREEN_THRESHOLD_PA / 100.0
SURFACE_PRESSURE_VARIABLE = "surface_pressure"

SCREENED_EVENT_FEATURES_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_event_features_sp900hpa.csv"
SCREENED_EVENT_FEATURES_PARTIAL_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_event_features_sp900hpa_partial.csv"
SCREENED_SWITCHING_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_switching_summary_sp900hpa.csv"
SCREENED_CROSSTAB_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_cluster_crosstab_sp900hpa.csv"
SCREENED_COUNTS_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_cluster_counts_sp900hpa.csv"
SCREENED_MEDIANS_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_cluster_medians_sp900hpa.csv"
SCREENED_SOLUTION_SUMMARY_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_solution_summary_sp900hpa.csv"
SCREENED_QUALITY_SCAN_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_quality_scan_sp900hpa.csv"
SCREENED_VARIANCE_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_pca_variance_sp900hpa.csv"
SCREENED_LOADINGS_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_pca_loadings_sp900hpa.csv"
SCREENED_DRIVERS_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_pca_drivers_sp900hpa.csv"
SCREENED_CLUSTERED_EVENTS_PATH = SENSITIVITY_EXPORT_DIR / "clustered_events_k3_screened_t850_sp900hpa.csv"
SCREENED_FEATURE_COMPARISON_PATH = SENSITIVITY_EXPORT_DIR / "k3_screened_t850_feature_comparison_by_current_cluster_sp900hpa.csv"

PCA_SCATTER_PATH = PLOT_DIR / "k3_pca_scatter_full_vs_screened_t850_sp900hpa.png"
PCA_VARIANCE_PLOT_PATH = PLOT_DIR / "k3_pca_variance_full_vs_screened_t850_sp900hpa.png"
T850_COMPARE_PLOT_PATH = PLOT_DIR / "k3_t850_unmasked_vs_screened_cluster_boxplots_sp900hpa.png"

FEATURE_LABELS = {
    "coastal_to_jpcz_mean_divergence_ratio": "Coastal/JPCZ signed-divergence ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12": "Hokkaido minimum z850 anomaly",
    DROPPED_FEATURE: "Front-box maximum |grad T850|",
    SCREENED_FEATURE_COLUMN: "Front-box maximum |grad T850| (surface pressure screened >= 900 hPa)",
    "sea_of_japan_mean_vorticity_peak_925": "Sea of Japan mean 925 hPa vorticity",
}
FEATURE_UNITS = {
    "coastal_to_jpcz_mean_divergence_ratio": "unitless",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12": "gpm",
    DROPPED_FEATURE: "K (100 km)^-1",
    SCREENED_FEATURE_COLUMN: "K (100 km)^-1",
    "sea_of_japan_mean_vorticity_peak_925": "1e-5 s^-1",
}


def maybe_copy_to_drive(path: Path, *, verbose: bool = True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True


def axis_label(column_name: str) -> str:
    label = FEATURE_LABELS.get(column_name, column_name)
    units = FEATURE_UNITS.get(column_name, "")
    if units and units != "unitless":
        return f"{label}\n[{units}]"
    return label


def cluster_count_table(labels: pd.Series) -> pd.DataFrame:
    counts = labels.value_counts().sort_index()
    max_count = int(counts.max())
    rows = []
    for cluster_id, n_events in counts.items():
        if n_events == max_count:
            size_rank = 1
            size_descriptor = "largest subgroup"
        elif n_events == int(counts.min()):
            size_rank = int((counts.rank(method="dense", ascending=False)).loc[cluster_id])
            size_descriptor = "smallest subgroup"
        else:
            size_rank = int((counts.rank(method="dense", ascending=False)).loc[cluster_id])
            size_descriptor = "second-largest subgroup" if size_rank == 2 else f"rank {size_rank} subgroup"
        rows.append(
            {
                "cluster_id": int(cluster_id),
                "n_events": int(n_events),
                "size_rank": size_rank,
                "size_descriptor": size_descriptor,
                "cluster_label": f"Cluster {int(cluster_id)}: n={int(n_events)} ({size_descriptor})",
            }
        )
    return pd.DataFrame(rows)


def best_cluster_label_mapping(reference_labels: pd.Series, candidate_labels: pd.Series):
    reference = reference_labels.astype(int)
    candidate = candidate_labels.astype(int).reindex(reference.index)
    unique_reference = sorted(reference.dropna().unique().tolist())
    unique_candidate = sorted(candidate.dropna().unique().tolist())
    if len(unique_reference) != len(unique_candidate):
        raise ValueError(
            "Expected the same number of clusters in the reference and candidate solutions, "
            f"got {unique_reference} versus {unique_candidate}."
        )

    best_mapping = None
    best_matches = -1
    for perm in permutations(unique_reference, len(unique_candidate)):
        mapping = dict(zip(unique_candidate, perm))
        mapped = candidate.map(mapping)
        match_count = int((mapped == reference).sum())
        if match_count > best_matches:
            best_matches = match_count
            best_mapping = mapping
    return best_mapping, best_matches


def compute_pca_diagnostics(feature_df: pd.DataFrame, feature_columns: list[str]):
    standardized_df, feature_means, feature_stds = standardize_feature_table(feature_df.copy(), columns=feature_columns)
    valid = standardized_df.dropna(axis=0, how="any")
    if valid.empty:
        raise ValueError("No complete rows are available for PCA.")

    matrix = valid.to_numpy(dtype=float)
    _, singular_values, vt = np.linalg.svd(matrix, full_matrices=False)
    n_components = min(3, matrix.shape[1])
    components = vt[:n_components]
    scores = matrix @ components.T

    total_variance = float((singular_values ** 2).sum())
    explained_variance_ratio = (singular_values[:n_components] ** 2) / total_variance

    score_df = pd.DataFrame(
        scores,
        index=valid.index,
        columns=[f"PC{i + 1}" for i in range(n_components)],
    )
    variance_df = pd.DataFrame(
        {
            "principal_component": [f"PC{i + 1}" for i in range(n_components)],
            "explained_variance_ratio": explained_variance_ratio,
            "explained_variance_percent": explained_variance_ratio * 100.0,
            "cumulative_explained_variance_ratio": np.cumsum(explained_variance_ratio),
            "cumulative_explained_variance_percent": np.cumsum(explained_variance_ratio) * 100.0,
        }
    )
    loadings_df = pd.DataFrame(
        components.T,
        index=feature_columns,
        columns=[f"PC{i + 1}" for i in range(n_components)],
    ).reset_index().rename(columns={"index": "feature"})
    loadings_long_df = loadings_df.melt(id_vars="feature", var_name="principal_component", value_name="loading")
    loadings_long_df["feature_label"] = loadings_long_df["feature"].map(FEATURE_LABELS)

    driver_rows = []
    for pc_name in variance_df["principal_component"]:
        component_subset = loadings_long_df.loc[loadings_long_df["principal_component"] == pc_name].copy()
        component_subset["abs_loading"] = component_subset["loading"].abs()
        top_two = component_subset.sort_values("abs_loading", ascending=False).head(2).reset_index(drop=True)
        driver_rows.append(
            {
                "principal_component": pc_name,
                "explained_variance_percent": float(
                    variance_df.loc[variance_df["principal_component"] == pc_name, "explained_variance_percent"].iloc[0]
                ),
                "top_driver_feature": top_two.loc[0, "feature"],
                "top_driver_label": top_two.loc[0, "feature_label"],
                "top_driver_loading": float(top_two.loc[0, "loading"]),
                "second_driver_feature": top_two.loc[1, "feature"] if len(top_two) > 1 else "",
                "second_driver_label": top_two.loc[1, "feature_label"] if len(top_two) > 1 else "",
                "second_driver_loading": float(top_two.loc[1, "loading"]) if len(top_two) > 1 else np.nan,
            }
        )
    driver_df = pd.DataFrame(driver_rows)
    return standardized_df, score_df, variance_df, loadings_long_df, driver_df, feature_means, feature_stds


def _box_max(field: xr.DataArray, box) -> float:
    boxed = field.sel(
        longitude=slice(box.lon_min, box.lon_max),
        latitude=slice(box.lat_max, box.lat_min),
    )
    return float(boxed.max(skipna=True).values)


def safe_nanmax(values) -> float:
    array = np.asarray(values, dtype=float)
    if array.size == 0 or np.all(np.isnan(array)):
        return float("nan")
    return float(np.nanmax(array))


def strip_nonspatial_coords(field: xr.DataArray) -> xr.DataArray:
    drop_coords = [coord_name for coord_name in field.coords if coord_name not in {"latitude", "longitude"}]
    if drop_coords:
        field = field.reset_coords(names=drop_coords, drop=True)
    return field


def align_mask_to_field(mask_field: xr.DataArray, target_field: xr.DataArray) -> xr.DataArray:
    aligned = mask_field.astype(float).interp(
        latitude=target_field.latitude,
        longitude=target_field.longitude,
        method="nearest",
    )
    return (aligned >= 0.5).rename(mask_field.name or "aligned_mask")


def normalize_surface_pressure_to_pa(surface_pressure_field: xr.DataArray) -> xr.DataArray:
    pressure_field = strip_nonspatial_coords(surface_pressure_field).astype(float)
    units_text = str(pressure_field.attrs.get("units", "")).strip().lower()
    finite_values = pressure_field.values[np.isfinite(pressure_field.values)]
    inferred_max = float(np.nanmax(finite_values)) if finite_values.size else float("nan")

    if units_text in {"hpa", "mb", "millibar", "millibars"}:
        scale_factor = 100.0
        unit_interpretation = units_text or "hPa"
    elif units_text in {"pa", "pascal", "pascals"}:
        scale_factor = 1.0
        unit_interpretation = units_text or "Pa"
    elif np.isfinite(inferred_max) and inferred_max < 2000.0:
        scale_factor = 100.0
        unit_interpretation = "inferred_hPa"
    else:
        scale_factor = 1.0
        unit_interpretation = units_text or "inferred_Pa"

    normalized = (pressure_field * scale_factor).rename(pressure_field.name or SURFACE_PRESSURE_VARIABLE)
    normalized.attrs = dict(pressure_field.attrs)
    normalized.attrs["units"] = "Pa"
    normalized.attrs["original_units"] = pressure_field.attrs.get("units", "")
    normalized.attrs["unit_interpretation"] = unit_interpretation
    normalized.attrs["scale_factor_to_pa"] = scale_factor
    return normalized


def build_surface_pressure_screen(surface_pressure_field: xr.DataArray) -> xr.DataArray:
    surface_pressure_pa = normalize_surface_pressure_to_pa(surface_pressure_field)
    screen = (surface_pressure_pa >= PRESSURE_SCREEN_THRESHOLD_PA).rename("surface_pressure_screen_mask")
    screen.attrs["threshold_pa"] = PRESSURE_SCREEN_THRESHOLD_PA
    screen.attrs["threshold_hpa"] = PRESSURE_SCREEN_THRESHOLD_HPA
    screen.attrs["surface_pressure_units"] = surface_pressure_pa.attrs.get("units", "")
    screen.attrs["surface_pressure_unit_interpretation"] = surface_pressure_pa.attrs.get("unit_interpretation", "")
    screen.attrs["meaning"] = "True where local surface pressure is at or above the 900 hPa T850 screening threshold"
    return screen


def ensure_surface_pressure_available(ds: xr.Dataset):
    if SURFACE_PRESSURE_VARIABLE in ds.data_vars:
        return
    surface_like = sorted([name for name in ds.data_vars if "surface" in name.lower() or name.lower() in {"sp", "msl"}])
    raise RuntimeError(
        "surface_pressure is not available in the opened ARCO ERA5 analysis-ready dataset. "
        + f"Surface-like variables found instead: {surface_like}"
    )


In [ ]:
for path_to_restore in [CLUSTERED_K3_PATH]:
    if not path_to_restore.exists():
        restore_from_drive_cache(CLUSTERED_K3_PATH)

clustered_k3_df = pd.read_csv(CLUSTERED_K3_PATH)
clustered_k3_df = add_japan_local_time_columns(clustered_k3_df)
if PRIMARY_CLUSTER_COLUMN not in clustered_k3_df.columns:
    cluster_cols = [column for column in clustered_k3_df.columns if column.startswith("cluster_")]
    raise RuntimeError(f"Expected {PRIMARY_CLUSTER_COLUMN} in clustered_events_k3.csv, found {cluster_cols}")

current_cluster_labels = clustered_k3_df[PRIMARY_CLUSTER_COLUMN].astype(int)
current_cluster_counts_df = cluster_count_table(current_cluster_labels)
current_cluster_label_lookup = current_cluster_counts_df.set_index("cluster_id")["cluster_label"].to_dict()

print("Loaded current clustered k=3 rerun table from Notebook 08 outputs")
display(current_cluster_counts_df)
print("\nThis notebook keeps the current 4-feature solution as the baseline and replaces only the T850 frontality metric with a surface-pressure-screened version.")


In [ ]:
if 'era5_runtime_ds' not in globals():
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})
ensure_surface_pressure_available(era5_runtime_ds)

sample_event_row = clustered_k3_df.iloc[0].copy()
sample_surface_snapshot = load_offset_snapshot(
    era5_runtime_ds,
    sample_event_row["event_peak"],
    offset_hours=0,
    variables=[SURFACE_PRESSURE_VARIABLE],
    domain=OBJECTIVE_SUBTYPE_DOMAIN,
    level=None,
)
sample_surface_pressure = normalize_surface_pressure_to_pa(sample_surface_snapshot[SURFACE_PRESSURE_VARIABLE])
sample_surface_screen = build_surface_pressure_screen(sample_surface_pressure)

surface_summary_df = pd.DataFrame(
    {
        "metric": [
            "surface pressure variable",
            "surface pressure original units",
            "surface pressure interpreted as",
            "screen threshold [Pa]",
            "screen threshold [hPa]",
            "sample event peak",
            "sample surface-pressure minimum [hPa]",
            "sample surface-pressure maximum [hPa]",
            "fraction of full domain kept by screen",
            "fraction of Hokkaido front box kept by screen",
        ],
        "value": [
            SURFACE_PRESSURE_VARIABLE,
            sample_surface_pressure.attrs.get("original_units", ""),
            sample_surface_pressure.attrs.get("unit_interpretation", ""),
            int(PRESSURE_SCREEN_THRESHOLD_PA),
            int(PRESSURE_SCREEN_THRESHOLD_HPA),
            str(pd.Timestamp(sample_event_row["event_peak"])),
            round(float(sample_surface_pressure.min().values) / 100.0, 2),
            round(float(sample_surface_pressure.max().values) / 100.0, 2),
            round(float(sample_surface_screen.mean().values), 3),
            round(
                float(
                    sample_surface_screen.sel(
                        longitude=slice(HOKKAIDO_FRONT_BOX.lon_min, HOKKAIDO_FRONT_BOX.lon_max),
                        latitude=slice(HOKKAIDO_FRONT_BOX.lat_max, HOKKAIDO_FRONT_BOX.lat_min),
                    ).mean().values
                ),
                3,
            ),
        ],
    }
)
print("Surface-pressure screening summary")
display(surface_summary_df)


In [ ]:
if 'surface_summary_df' not in globals() or 'clustered_k3_df' not in globals():
    raise RuntimeError(
        "Run the surface-pressure screening summary cell first so the screen and clustered table are defined before the sanity check."
    )

if 'era5_runtime_ds' not in globals():
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})

sample_event_row = clustered_k3_df.iloc[0].copy()
sample_temp_snapshot_850 = load_offset_snapshot(
    era5_runtime_ds,
    sample_event_row["event_peak"],
    offset_hours=0,
    variables=["temperature"],
    domain=OBJECTIVE_SUBTYPE_DOMAIN,
    level=850,
)
sample_surface_snapshot = load_offset_snapshot(
    era5_runtime_ds,
    sample_event_row["event_peak"],
    offset_hours=0,
    variables=[SURFACE_PRESSURE_VARIABLE],
    domain=OBJECTIVE_SUBTYPE_DOMAIN,
    level=None,
)
sample_temp_grad = compute_temperature_gradient_magnitude(sample_temp_snapshot_850)
sample_temp_grad_display = (sample_temp_grad * float(sample_temp_grad.attrs.get("display_scale_factor", 1.0))).rename("temperature_gradient_display")
sample_surface_pressure = normalize_surface_pressure_to_pa(sample_surface_snapshot[SURFACE_PRESSURE_VARIABLE])
sample_pressure_screen = build_surface_pressure_screen(sample_surface_pressure)
sample_aligned_screen = align_mask_to_field(sample_pressure_screen, sample_temp_grad_display)
sample_screened_temp_grad = sample_temp_grad_display.where(sample_aligned_screen)

sample_front_box_unmasked = sample_temp_grad_display.sel(
    longitude=slice(HOKKAIDO_FRONT_BOX.lon_min, HOKKAIDO_FRONT_BOX.lon_max),
    latitude=slice(HOKKAIDO_FRONT_BOX.lat_max, HOKKAIDO_FRONT_BOX.lat_min),
)
sample_front_box_screened = sample_screened_temp_grad.sel(
    longitude=slice(HOKKAIDO_FRONT_BOX.lon_min, HOKKAIDO_FRONT_BOX.lon_max),
    latitude=slice(HOKKAIDO_FRONT_BOX.lat_max, HOKKAIDO_FRONT_BOX.lat_min),
)

sample_sanity_df = pd.DataFrame(
    {
        "metric": [
            "sample event peak",
            "sample current cluster",
            "sample offset hours",
            "surface pressure interpreted as",
            "aligned screen full-domain keep fraction",
            "aligned screen Hokkaido front-box keep fraction",
            "sample front-box unmasked max [K (100 km)^-1]",
            "sample front-box screened max [K (100 km)^-1]",
            "sample front-box finite-cell count after screening",
        ],
        "value": [
            str(pd.Timestamp(sample_event_row["event_peak"])),
            int(sample_event_row[PRIMARY_CLUSTER_COLUMN]),
            0,
            sample_surface_pressure.attrs.get("unit_interpretation", ""),
            round(float(sample_aligned_screen.mean().values), 3),
            round(float(sample_front_box_screened.notnull().mean().values), 3),
            round(float(sample_front_box_unmasked.max(skipna=True).values), 3),
            round(float(sample_front_box_screened.max(skipna=True).values), 3),
            int(sample_front_box_screened.notnull().sum().values),
        ],
    }
)
print("One-event pressure-screened T850 sanity check before the full event loop")
display(sample_sanity_df)
if int(sample_front_box_screened.notnull().sum().values) <= 0:
    raise RuntimeError(
        "The sanity-check event has zero valid screened cells in the Hokkaido front box. "
        "Do not run the full screened-feature loop until the pressure threshold or source is corrected."
    )


In [ ]:
for path_to_restore in [SCREENED_EVENT_FEATURES_PATH, SCREENED_EVENT_FEATURES_PARTIAL_PATH]:
    if not path_to_restore.exists():
        restore_from_drive_cache(path_to_restore)

if 'era5_runtime_ds' not in globals():
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})
ensure_surface_pressure_available(era5_runtime_ds)

existing_screened_df = None
if SCREENED_EVENT_FEATURES_PATH.exists() and not FORCE_REBUILD_SCREENED_T850:
    existing_screened_df = pd.read_csv(SCREENED_EVENT_FEATURES_PATH)
    print("Loaded cached final pressure-screened T850 event feature table")
elif SCREENED_EVENT_FEATURES_PARTIAL_PATH.exists() and not FORCE_REBUILD_SCREENED_T850:
    existing_screened_df = pd.read_csv(SCREENED_EVENT_FEATURES_PARTIAL_PATH)
    print("Loaded cached partial pressure-screened T850 event feature table")

if existing_screened_df is not None and not existing_screened_df.empty:
    existing_screened_df["event_row_index"] = pd.to_numeric(existing_screened_df["event_row_index"], errors="coerce").astype("Int64")
    existing_screened_df[SCREENED_FEATURE_COLUMN] = pd.to_numeric(existing_screened_df[SCREENED_FEATURE_COLUMN], errors="coerce")
    finite_cached = int(existing_screened_df[SCREENED_FEATURE_COLUMN].notna().sum())
    print(f"Cached pressure-screened T850 rows with finite feature values: {finite_cached}/{len(existing_screened_df)}")
    if finite_cached == 0:
        print("Cached pressure-screened T850 table has zero finite values, so it will be ignored and rebuilt.")
        existing_screened_df = None

processed_records = []
processed_event_indices = set()
if existing_screened_df is not None and not existing_screened_df.empty:
    processed_records = existing_screened_df.to_dict(orient="records")
    processed_event_indices = {int(value) for value in existing_screened_df["event_row_index"].dropna().tolist()}


def compute_screened_t850_feature_for_event(row: pd.Series) -> float:
    feature_values = []
    for offset_hours in OFFSET_HOURS:
        temp_snapshot_850 = load_offset_snapshot(
            era5_runtime_ds,
            row["event_peak"],
            offset_hours=offset_hours,
            variables=["temperature"],
            domain=OBJECTIVE_SUBTYPE_DOMAIN,
            level=850,
        )
        surface_snapshot = load_offset_snapshot(
            era5_runtime_ds,
            row["event_peak"],
            offset_hours=offset_hours,
            variables=[SURFACE_PRESSURE_VARIABLE],
            domain=OBJECTIVE_SUBTYPE_DOMAIN,
            level=None,
        )
        temp_grad = compute_temperature_gradient_magnitude(temp_snapshot_850)
        temp_grad_display = (temp_grad * float(temp_grad.attrs.get("display_scale_factor", 1.0))).rename("temperature_gradient_display")
        surface_pressure = normalize_surface_pressure_to_pa(surface_snapshot[SURFACE_PRESSURE_VARIABLE])
        pressure_screen = build_surface_pressure_screen(surface_pressure)
        aligned_screen = align_mask_to_field(pressure_screen, temp_grad_display)
        screened_temp_grad = temp_grad_display.where(aligned_screen)
        feature_values.append(_box_max(screened_temp_grad, HOKKAIDO_FRONT_BOX))
    return safe_nanmax(feature_values)

remaining_rows = clustered_k3_df.loc[~clustered_k3_df.index.isin(processed_event_indices)].copy()
print(f"Pressure-screened T850 feature progress before this run: {len(processed_event_indices)}/{len(clustered_k3_df)} events")

for _, (row_index, row) in enumerate(remaining_rows.iterrows(), start=1):
    screened_feature_value = compute_screened_t850_feature_for_event(row)
    processed_records.append(
        {
            "event_row_index": int(row_index),
            "event_peak": pd.Timestamp(row["event_peak"]),
            "current_cluster_id": int(row[PRIMARY_CLUSTER_COLUMN]),
            "current_cluster_label": current_cluster_label_lookup[int(row[PRIMARY_CLUSTER_COLUMN])],
            "unmasked_t850_feature": float(row[DROPPED_FEATURE]),
            SCREENED_FEATURE_COLUMN: screened_feature_value,
            "screened_minus_unmasked_t850": screened_feature_value - float(row[DROPPED_FEATURE]) if np.isfinite(screened_feature_value) else np.nan,
        }
    )

    event_number = len(processed_records)
    if event_number % CHECKPOINT_EVERY_EVENTS == 0 or event_number == len(clustered_k3_df):
        checkpoint_df = pd.DataFrame(processed_records).sort_values("event_row_index").reset_index(drop=True)
        checkpoint_df.to_csv(SCREENED_EVENT_FEATURES_PARTIAL_PATH, index=False)
        maybe_copy_to_drive(SCREENED_EVENT_FEATURES_PARTIAL_PATH)
        print(f"Checkpointed pressure-screened T850 event feature table at {event_number}/{len(clustered_k3_df)} events")

screened_event_feature_df = pd.DataFrame(processed_records).sort_values("event_row_index").reset_index(drop=True)
screened_event_feature_df["event_row_index"] = pd.to_numeric(screened_event_feature_df["event_row_index"], errors="coerce").astype(int)
screened_event_feature_df[SCREENED_FEATURE_COLUMN] = pd.to_numeric(screened_event_feature_df[SCREENED_FEATURE_COLUMN], errors="coerce")
finite_screened_events = int(screened_event_feature_df[SCREENED_FEATURE_COLUMN].notna().sum())
print(f"Finite pressure-screened T850 feature values available after this run: {finite_screened_events}/{len(screened_event_feature_df)} events")
if finite_screened_events == 0:
    raise RuntimeError(
        "The pressure-screened T850 feature table contains zero finite values. "
        "This usually means the pressure threshold is too strict or the surface-pressure field did not align correctly with the temperature grid."
    )
screened_event_feature_df.to_csv(SCREENED_EVENT_FEATURES_PATH, index=False)
maybe_copy_to_drive(SCREENED_EVENT_FEATURES_PATH)
print("Saved final pressure-screened T850 event feature table")

screened_feature_comparison_df = (
    screened_event_feature_df.groupby("current_cluster_id")
    .agg(
        cluster_label=("current_cluster_label", "first"),
        n_events=("event_row_index", "count"),
        unmasked_t850_median=("unmasked_t850_feature", "median"),
        screened_t850_median=(SCREENED_FEATURE_COLUMN, "median"),
        median_screened_minus_unmasked=("screened_minus_unmasked_t850", "median"),
        mean_abs_screened_minus_unmasked=("screened_minus_unmasked_t850", lambda s: float(np.nanmean(np.abs(s)))),
    )
    .reset_index()
    .rename(columns={"current_cluster_id": "cluster_id"})
)
for column_name in ["unmasked_t850_median", "screened_t850_median", "median_screened_minus_unmasked", "mean_abs_screened_minus_unmasked"]:
    screened_feature_comparison_df[column_name] = screened_feature_comparison_df[column_name].round(3)
screened_feature_comparison_df.to_csv(SCREENED_FEATURE_COMPARISON_PATH, index=False)
maybe_copy_to_drive(SCREENED_FEATURE_COMPARISON_PATH)

print("Per-current-cluster comparison of unmasked versus pressure-screened T850 feature")
display(screened_feature_comparison_df)


In [ ]:
baseline_standardized_df, baseline_scores_df, baseline_variance_df, baseline_loadings_df, baseline_driver_df, baseline_feature_means, baseline_feature_stds = compute_pca_diagnostics(
    clustered_k3_df,
    FULL_FEATURE_COLUMNS,
)

screened_feature_df = clustered_k3_df.copy()
screened_event_feature_df["event_row_index"] = pd.to_numeric(screened_event_feature_df["event_row_index"], errors="coerce").astype(int)
screened_lookup = screened_event_feature_df.set_index("event_row_index")[SCREENED_FEATURE_COLUMN]
screened_feature_df[SCREENED_FEATURE_COLUMN] = pd.to_numeric(screened_lookup.reindex(screened_feature_df.index), errors="coerce")
finite_screened_feature_rows = int(screened_feature_df[SCREENED_FEATURE_COLUMN].notna().sum())
print(f"Rows with finite pressure-screened T850 values entering PCA/clustering: {finite_screened_feature_rows}/{len(screened_feature_df)}")
if finite_screened_feature_rows == 0:
    raise RuntimeError(
        "No finite pressure-screened T850 rows are available for PCA/clustering. "
        "Rerun the summary, sanity-check, and pressure-screened feature build cells with the updated logic."
    )

screened_standardized_df, screened_scores_df, screened_variance_df, screened_loadings_df, screened_driver_df, screened_feature_means, screened_feature_stds = compute_pca_diagnostics(
    screened_feature_df,
    SCREENED_FEATURE_COLUMNS,
)

screened_quality_df = evaluate_hierarchical_cluster_solutions(
    screened_standardized_df,
    cluster_counts=CLUSTER_COUNT_OPTIONS,
    method="ward",
)
screened_cluster_labels_raw = assign_hierarchical_clusters(
    screened_standardized_df,
    n_clusters=PRIMARY_CLUSTER_K,
    method="ward",
).rename(SCREENED_CLUSTER_COLUMN_RAW)
label_mapping, best_match_count = best_cluster_label_mapping(current_cluster_labels.loc[screened_cluster_labels_raw.index], screened_cluster_labels_raw)
screened_cluster_labels = screened_cluster_labels_raw.map(label_mapping).rename(SCREENED_CLUSTER_COLUMN)

screened_feature_df[SCREENED_CLUSTER_COLUMN_RAW] = screened_cluster_labels_raw.reindex(screened_feature_df.index)
screened_feature_df[SCREENED_CLUSTER_COLUMN] = screened_cluster_labels.reindex(screened_feature_df.index)

switch_mask = screened_cluster_labels != current_cluster_labels.loc[screened_cluster_labels.index]
switching_rows = [
    {
        "cluster_id": "all",
        "cluster_label": "All events",
        "n_events": int(len(screened_cluster_labels)),
        "n_switched": int(switch_mask.sum()),
        "percent_switched": float(100.0 * switch_mask.mean()),
        "best_label_mapping": str(label_mapping),
        "best_label_match_count": int(best_match_count),
    }
]
for cluster_id in sorted(current_cluster_labels.dropna().astype(int).unique()):
    cluster_mask = current_cluster_labels.loc[screened_cluster_labels.index] == int(cluster_id)
    switching_rows.append(
        {
            "cluster_id": int(cluster_id),
            "cluster_label": current_cluster_label_lookup[int(cluster_id)],
            "n_events": int(cluster_mask.sum()),
            "n_switched": int(switch_mask.loc[cluster_mask].sum()),
            "percent_switched": float(100.0 * switch_mask.loc[cluster_mask].mean()),
            "best_label_mapping": str(label_mapping),
            "best_label_match_count": int(best_match_count),
        }
    )
switching_df = pd.DataFrame(switching_rows)
switching_df["percent_switched"] = switching_df["percent_switched"].round(2)

crosstab_df = pd.crosstab(
    current_cluster_labels.rename("current_4_feature_cluster"),
    screened_cluster_labels.rename("screened_t850_cluster"),
)
screened_cluster_counts_df = cluster_count_table(screened_cluster_labels)

screened_cluster_medians_df = (
    screened_feature_df.loc[screened_cluster_labels.index]
    .groupby(screened_cluster_labels)[[
        "coastal_to_jpcz_mean_divergence_ratio",
        "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
        DROPPED_FEATURE,
        SCREENED_FEATURE_COLUMN,
        "sea_of_japan_mean_vorticity_peak_925",
    ]]
    .median()
    .round(3)
    .reset_index()
    .rename(columns={SCREENED_CLUSTER_COLUMN: "cluster_id"})
)

solution_summary_df = pd.DataFrame(
    [
        {
            "solution": "current_4_feature_k3",
            "n_features": len(FULL_FEATURE_COLUMNS),
            "features": ", ".join(FULL_FEATURE_COLUMNS),
            "silhouette": float(compute_mean_silhouette_score(baseline_standardized_df, current_cluster_labels)),
            "cluster_counts": ", ".join(
                [f"{int(row.cluster_id)}:{int(row.n_events)}" for row in current_cluster_counts_df.itertuples(index=False)]
            ),
        },
        {
            "solution": "screened_t850_sp900hpa_k3",
            "n_features": len(SCREENED_FEATURE_COLUMNS),
            "features": ", ".join(SCREENED_FEATURE_COLUMNS),
            "silhouette": float(compute_mean_silhouette_score(screened_standardized_df, screened_cluster_labels)),
            "cluster_counts": ", ".join(
                [f"{int(row.cluster_id)}:{int(row.n_events)}" for row in screened_cluster_counts_df.itertuples(index=False)]
            ),
        },
    ]
)
solution_summary_df["silhouette"] = solution_summary_df["silhouette"].round(5)

combined_variance_df = pd.concat(
    [
        baseline_variance_df.assign(solution="current_4_feature_k3"),
        screened_variance_df.assign(solution="screened_t850_sp900hpa_k3"),
    ],
    ignore_index=True,
)
combined_loadings_df = pd.concat(
    [
        baseline_loadings_df.assign(solution="current_4_feature_k3"),
        screened_loadings_df.assign(solution="screened_t850_sp900hpa_k3"),
    ],
    ignore_index=True,
)
combined_driver_df = pd.concat(
    [
        baseline_driver_df.assign(solution="current_4_feature_k3"),
        screened_driver_df.assign(solution="screened_t850_sp900hpa_k3"),
    ],
    ignore_index=True,
)

output_tables = {
    SCREENED_SWITCHING_PATH.name: switching_df,
    SCREENED_CROSSTAB_PATH.name: crosstab_df.reset_index(),
    SCREENED_COUNTS_PATH.name: screened_cluster_counts_df,
    SCREENED_MEDIANS_PATH.name: screened_cluster_medians_df,
    SCREENED_SOLUTION_SUMMARY_PATH.name: solution_summary_df,
    SCREENED_QUALITY_SCAN_PATH.name: screened_quality_df,
    SCREENED_VARIANCE_PATH.name: combined_variance_df,
    SCREENED_LOADINGS_PATH.name: combined_loadings_df,
    SCREENED_DRIVERS_PATH.name: combined_driver_df,
    SCREENED_CLUSTERED_EVENTS_PATH.name: screened_feature_df,
}
for filename, table_df in output_tables.items():
    output_path = SENSITIVITY_EXPORT_DIR / filename
    table_df.to_csv(output_path, index=False)
    maybe_copy_to_drive(output_path)

print("Current 4-feature versus surface-pressure-screened T850 k=3 comparison")
display(solution_summary_df)
print("\nSurface-pressure-screened 4-feature quality scan across k = 2, 3, 4")
display(screened_quality_df)
print("\nPercent of events that switch clusters after replacing T850 with the surface-pressure-screened >=900 hPa version and best-matching the labels")
display(switching_df)
print("\nCurrent 4-feature versus surface-pressure-screened cluster cross-tab")
display(crosstab_df)
print("\nSurface-pressure-screened cluster medians, including both the original and screened T850 feature columns for comparison")
display(screened_cluster_medians_df)
print("\nPCA variance comparison")
display(combined_variance_df)
print("\nTop PCA loading drivers")
display(combined_driver_df)


In [ ]:
cluster_colors = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
plot_specs = [
    (
        axes[0],
        baseline_scores_df,
        current_cluster_labels.loc[baseline_scores_df.index].astype(int),
        baseline_variance_df,
        "Current 4-feature PCA space",
    ),
    (
        axes[1],
        screened_scores_df,
        screened_cluster_labels.loc[screened_scores_df.index].astype(int),
        screened_variance_df,
        "Pressure-screened T850 PCA space (surface pressure >= 900 hPa)",
    ),
]

for ax, score_df, labels, variance_df, title in plot_specs:
    for cluster_id in sorted(labels.unique()):
        mask = labels == cluster_id
        ax.scatter(
            score_df.loc[mask, "PC1"],
            score_df.loc[mask, "PC2"],
            s=28,
            alpha=0.8,
            color=cluster_colors.get(int(cluster_id), "gray"),
            label=f"Cluster {int(cluster_id)}",
        )
        centroid_x = float(score_df.loc[mask, "PC1"].mean())
        centroid_y = float(score_df.loc[mask, "PC2"].mean())
        ax.text(centroid_x, centroid_y, f"C{int(cluster_id)}", fontsize=10, weight="bold")
    pc1_pct = float(variance_df.loc[variance_df["principal_component"] == "PC1", "explained_variance_percent"].iloc[0])
    pc2_pct = float(variance_df.loc[variance_df["principal_component"] == "PC2", "explained_variance_percent"].iloc[0])
    ax.set_xlabel(f"PC1 ({pc1_pct:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({pc2_pct:.1f}% variance)")
    ax.set_title(title)
    ax.grid(alpha=0.25)
    ax.legend(loc="best")

fig.suptitle(
    "PCA comparison: current 4-feature solution versus surface-pressure-screened T850 solution",
    fontsize=13,
)
if SAVE_PLOTS:
    fig.savefig(PCA_SCATTER_PATH, dpi=180, bbox_inches="tight")
    maybe_copy_to_drive(PCA_SCATTER_PATH, verbose=False)
plt.show()

variance_fig, variance_ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
variance_pivot = combined_variance_df.pivot(index="principal_component", columns="solution", values="explained_variance_percent")
variance_pivot.plot(kind="bar", ax=variance_ax)
variance_ax.set_ylabel("Explained variance [%]")
variance_ax.set_title("Explained variance comparison after pressure-screening T850 at surface pressure >= 900 hPa")
variance_ax.grid(axis="y", alpha=0.25)
if SAVE_PLOTS:
    variance_fig.savefig(PCA_VARIANCE_PLOT_PATH, dpi=180, bbox_inches="tight")
    maybe_copy_to_drive(PCA_VARIANCE_PLOT_PATH, verbose=False)
plt.show()

plot_feature_df = screened_event_feature_df.copy()
plot_feature_df["current_cluster_id"] = plot_feature_df["current_cluster_id"].astype(int)
plot_feature_df["current_cluster_label"] = plot_feature_df["current_cluster_id"].map(current_cluster_label_lookup)
plot_feature_df = plot_feature_df.sort_values("current_cluster_id")
positions = []
labels = []
boxplot_data = []
colors = []
for cluster_id in sorted(plot_feature_df["current_cluster_id"].unique()):
    cluster_subset = plot_feature_df.loc[plot_feature_df["current_cluster_id"] == cluster_id]
    positions.extend([cluster_id - 0.15, cluster_id + 0.15])
    labels.extend([f"C{cluster_id}\nunmasked", f"C{cluster_id}\nscreened"])
    boxplot_data.extend([
        cluster_subset["unmasked_t850_feature"].dropna().values,
        cluster_subset[SCREENED_FEATURE_COLUMN].dropna().values,
    ])
    colors.extend(["#9ecae1", "#fdae6b"])

compare_fig, compare_ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
box = compare_ax.boxplot(boxplot_data, positions=positions, widths=0.22, patch_artist=True, showfliers=False)
for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
compare_ax.set_xticks(positions)
compare_ax.set_xticklabels(labels)
compare_ax.set_ylabel("Front-box maximum |grad T850| [K (100 km)^-1]")
compare_ax.set_title("Current-cluster comparison of unmasked versus surface-pressure-screened T850 feature")
compare_ax.grid(axis="y", alpha=0.25)
if SAVE_PLOTS:
    compare_fig.savefig(T850_COMPARE_PLOT_PATH, dpi=180, bbox_inches="tight")
    maybe_copy_to_drive(T850_COMPARE_PLOT_PATH, verbose=False)
plt.show()

print("Saved pressure-screened T850 sensitivity plots")
display(pd.DataFrame({
    "plot": ["PCA scatter", "PCA variance comparison", "Current-cluster T850 unmasked versus screened boxplots"],
    "path": [str(PCA_SCATTER_PATH), str(PCA_VARIANCE_PLOT_PATH), str(T850_COMPARE_PLOT_PATH)],
}))


### How to read the result

- If switching is small and the pressure-screened `T850` `k = 3` solution looks close to the current one, then the screen is cleaning the feature without breaking the subtype story.
- If switching stays large, or if the screened solution still collapses the current refined split, then the more conservative interpretation is that the broad `k = 2` structure is the more robust one.
- This notebook changes **only** the `T850` frontality metric so that the effect of the `surface_pressure >= 900 hPa` screen can be isolated cleanly.
